<!--
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Background on ATen ops

There is a third class of callables in PyTorch beyond functions and modules:
ops.

Behind the scenes of a call to a function like `torch.nn.functional.linear(a,
b)` or even one with a custom autograd function like `SigmoidNoVanishing`,
PyTorch converts these calls into ATen and c10d ops. ATen stands for "A Tensor
Library", and c10d is an abbreviation for caffe distributed. c10d ops are for
collectives to enable communication between devices. Everything else is an ATen
op. ATen contains thousands of ops, and these ops form an Intermediate
Representation (IR).

As you saw in the implementation of `torch.nn.Linear`, the forward method simply
calls `torch.nn.functional.linear`. Ultimately, that call will get translated by
this block of C++ into the aten ops of `addmm` and `.t` (transpose).

```cpp
return at::addmm(*bias, input, weight.t());
```

[PyTorch source](https://github.com/pytorch/pytorch/blob/5b4d57d8bb7dbe3ddb5dced469df4f5dd1e82a11/aten/src/ATen/native/Linear.cpp#L85)

You will now explore how to view the IR that the PyTorch IR engine creates
internally. Look at the sample implementation below. Notice that logs for aot are
enabled:

```python
torch._logging.set_logs(aot_graphs=True)
```

AOT stands for Ahead-Of-Time compilation. In this context, it basically means
the series of aten ops that your Python code was lowered into.

Inspecting the logs, you'll see the key lines: a transpose, then an addmm.

```log
t: "f32[64, 64][1, 64]cpu" = torch.ops.aten.t.default(primals_1);  primals_1 = None
addmm: "f32[32, 64][64, 1]cpu" = torch.ops.aten.addmm.default(primals_2, primals_3, t);  primals_2 = t = None
```

You now know how to view the ATen ops that PyTorch creates internally.

**Exercise for the reader**: Apply the same logging as in the sample below to **[Extending Modules via Composition](extending_modules_and_functions_via_composition.ipynb)** and figure
out which ATen ops correspond to the bit operations.

```python
  # Shift each of the 32 bits by zero to 31 positions.
  shifted = packed >> torch.arange(32, dtype=torch.uint32)
  # AKA shifted = packed.bitwise_right_shift(torch.arange(32, dtype=torch.uint32))

  # Mask out the upper bits.
  values = shifted & 1
  # AKA values = shifted.bitwise_and(1)
```

**Exercise for the reader**: Apply the same logging technique to **[Customizing Autograd via torch.autograd.Function](customizing_autograd_via_torch_autograd_function.ipynb)** and figure out
which ATen ops correspond to `torch.where` and the `>` and `<=` operator.

```python
local_grad = torch.where(
    torch.logical_and(result > 0.1, result <= 0.9),
    torch.tensor(0.25, device=result.device),
    torch.tensor(0.01, device=result.device),
)
```

*Answer*:

```
gt: "b8[7, 1][1, 1]cpu" = torch.ops.aten.gt.Scalar(sigmoid, 0.1)
le: "b8[7, 1][1, 1]cpu" = torch.ops.aten.le.Scalar(sigmoid, 0.9)
```

In [1]:
from absl import app as absl_app
import torch

torch._logging.set_logs(aot_graphs=True)  # pylint: disable=protected-access

Device type: tpu, Device index: default
Initializing TPU distributed runtime


I0000 00:00:1787340640.259580   76681 pjrt_api.cc:119] GetPjrtApi was found for TPU at /home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/libtpu/libtpu.so
I0000 00:00:1787340640.275897   76681 tpu_profiler_plugin.cc:791] PyTorch PrivateUse1ProfilerRegistry not found. Native TPU profiling will be disabled (this is expected on PyTorch < 2.12).
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
I0000 00:00:1787340640.297186   76681 device_rt.cc:130] PjRt runtime initialization deferred for tpu


In [2]:
@torch.compile(backend="aot_eager")
def fwd_bwd(layer_one, layer_two, data):
  output = layer_two(layer_one(data))
  output.sum().backward()

In [3]:
def main():
  fwd_bwd(
      layer_one=torch.nn.Linear(64, 64),
      layer_two=torch.nn.functional.relu,
      data=torch.randn(32, 64),
  )
  print("Success.")

main()

I0000 00:00:1787340640.322728   76681 pjrt_api.cc:168] The PJRT plugin has PJRT API version 0.114. The framework PJRT API version is 0.114.
E0821 19:30:40.359688   76681 hugepage_text.cc:344] RAW: File offset incorrectly aligned for file-backed THP: (59e44dc00000 & ~ffffffffffe00000) = 0 != 126000 = (126000 & ~ffffffffffe00000)
I0000 00:00:1787340641.655995   76681 pjrt_c_api_client.cc:188] PjRtCApiClient created.
V0821 19:30:41.782000 76681 site-packages/torch/_functorch/_aot_autograd/graph_capture.py:229] [0/0] [__aot_graphs] aot_config id: 0, fw_metadata=ViewAndMutationMeta(input_info=[InputAliasInfo(is_leaf=True, mutates_data=False, mutates_metadata=False, mutations_hidden_from_autograd=True, mutations_under_no_grad_or_inference_mode=False, mutation_inductor_storage_resize=False, mutates_storage_metadata=False, requires_grad=True, keep_input_mutations=True), InputAliasInfo(is_leaf=True, mutates_data=False, mutates_metadata=False, mutations_hidden_from_autograd=True, mutations_under

I0821 19:30:41.799000 76681 site-packages/torch/_functorch/_aot_autograd/graph_capture.py:322] [0/0] [__aot_graphs] TRACED GRAPH
I0821 19:30:41.799000 76681 site-packages/torch/_functorch/_aot_autograd/graph_capture.py:322] [0/0] [__aot_graphs]  ===== Forward graph 0 =====
I0821 19:30:41.799000 76681 site-packages/torch/_functorch/_aot_autograd/graph_capture.py:322] [0/0] [__aot_graphs]  /home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/torch/fx/_lazy_graph_module.py class <lambda>(torch.nn.Module):
I0821 19:30:41.799000 76681 site-packages/torch/_functorch/_aot_autograd/graph_capture.py:322] [0/0] [__aot_graphs]     def forward(self, arg0_1: "f32[64, 64][64, 1]cpu", arg1_1: "f32[64][1]cpu", arg2_1: "f32[32, 64][64, 1]cpu"):
I0821 19:30:41.799000 76681 site-packages/torch/_functorch/_aot_autograd/graph_capture.py:322] [0/0] [__aot_graphs]         # File: /tmp/ipykernel_76681/982231593.py:3 in fwd_bwd, code: output = layer_two(layer_one(data))
I0821 19:30:41.7990

Success.
